# Week 2 build · Rebuild micrograd, train it on XOR and moons

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mjtri/learnml/blob/main/notebooks/week-02.ipynb)

**~3 hours · CPU only · run top to bottom.**

**The one rule:** every experiment is preceded by a 🔮 **predict cell**. Type what you expect and how sure you are *before* running the experiment. The notebook refuses empty predictions. A rough guess is fine; the gap between guess and result is what you are here to collect (canvas steps 7–8: *predict before running*, *log the surprise*).

**This week's extra rule:** Part A is written **from memory**. Close the lessons first.

| Part | Topic | Time |
|---|---|---|
| A | `Value` from memory, then a PASS/FAIL test | 40 min |
| B | Every gradient checked three ways | 25 min |
| C | The accumulation bug, caused on purpose | 15 min |
| D | Neuron → Layer → MLP; XOR with and without the bend | 40 min |
| E | Moons: loss curve, decision boundary, PyTorch twin | 45 min |
| F | Stretch: ReLU vs tanh, learning-rate sweep | 15 min |

In [ ]:
#@title Setup: run me first
import numpy as np, torch, matplotlib.pyplot as plt
from IPython.display import Audio, Markdown, display
torch.manual_seed(0); np.random.seed(0)
_LEDGER = {}

def predict(tag, prediction, confidence):
    """Log a prediction BEFORE the experiment. Refuses empty predictions."""
    if not str(prediction).strip():
        raise ValueError(f"[{tag}] Write a prediction first. A rough guess is fine; an empty one is not.")
    _LEDGER[tag] = {"prediction": str(prediction).strip(), "confidence": int(confidence), "actual": "", "surprise": ""}
    print(f"Logged [{tag}] at {confidence}% confidence: {prediction}")

def record(tag, what_happened, surprise="none"):
    if tag not in _LEDGER:
        raise ValueError(f"[{tag}] has no prediction yet. Run its predict cell first.")
    _LEDGER[tag].update(actual=str(what_happened).strip(), surprise=surprise)
    print(f"Recorded [{tag}] surprise={surprise}")

def reveal():
    rows = ["| # | prediction | sure | what happened | surprise |", "|---|---|---|---|---|"]
    rows += [f"| {t} | {r['prediction']} | {r['confidence']}% | {r['actual']} | {r['surprise']} |" for t, r in _LEDGER.items()]
    print("\n".join(rows)); display(Markdown("\n".join(rows)))

print("torch", torch.__version__, "| ready")

In [ ]:
#@title Session size: SMOKE = quick CPU run (a few minutes); set False for the real session
SMOKE = True  #@param {type:"boolean"}
import random, math, time
random.seed(0)
STEPS_XOR  = 80 if SMOKE else 400     # full-batch steps on the 4 XOR points
STEPS_MOON = 40 if SMOKE else 300     # steps on the moons set
N_MOON     = 40 if SMOKE else 100     # points in the moons set
LR = 0.05
print("SMOKE" if SMOKE else "FULL", "| XOR steps", STEPS_XOR, "| moons:", N_MOON, "points,", STEPS_MOON, "steps")

---
## Part A · `Value` from memory (40 min)

**Words used in this part.** *Value object*: one number plus its history (`data`, `grad`, `_prev`, `_op`, `_backward`). *topological order*: parents before children; backward walks it reversed. *gradient accumulation*: closures write `+=`, so a value used twice collects both shares.

Fill in the class below **without looking at the lessons**: `__add__`, `__mul__`, `tanh`, `backward`. The test cell after it prints PASS or FAIL and never stops the notebook. If it fails, fix it, and note in the ledger which piece you had forgotten. The reference version further down is hidden inside a form cell; open it only after your test.

In [ ]:
class Value:
    """Your Value, from memory. The skeleton compiles; the four methods are yours."""
    def __init__(self, data, _children=(), _op=""):
        self.data, self.grad = float(data), 0.0
        self._prev, self._op = set(_children), _op
        self._backward = lambda: None
    def __repr__(self): return f"Value(data={self.data:.4g}, grad={self.grad:.4g})"
    # TODO 1: __add__(self, other) -> new Value; its _backward hands out.grad to both parents
    # TODO 2: __mul__(self, other) -> new Value; each parent gets the OTHER's data x out.grad
    # TODO 3: tanh(self)          -> new Value; parent gets (1 - t*t) x out.grad
    # TODO 4: backward(self)      -> topological order, self.grad = 1, walk it reversed

In [ ]:
#@title 🔮 Predict A1: Which of the four pieces (add, mul, tanh, backward, or 'used twice') is most likely to fail, and why?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("A1", prediction, confidence)

In [ ]:
def test_value(V):
    checks = []
    try:
        a, b, c = V(2.0), V(-3.0), V(10.0)
        e = a * b + c; L = e * e; L.backward()
        checks.append(("L = (a*b+c)^2 forward = 16", abs(L.data - 16) < 1e-9))
        checks.append(("a.grad = -24, b.grad = 16, c.grad = 8",
                       abs(a.grad + 24) < 1e-9 and abs(b.grad - 16) < 1e-9 and abs(c.grad - 8) < 1e-9))
        x = V(0.5); t = x.tanh(); t.backward()
        checks.append(("tanh(0.5) forward", abs(t.data - math.tanh(0.5)) < 1e-9))
        checks.append(("tanh backward = 1 - t^2", abs(x.grad - (1 - math.tanh(0.5) ** 2)) < 1e-9))
        y = V(3.0); z = y * y + y; z.backward()            # y used three times: 2y + 1 = 7
        checks.append(("value used in two places adds up (7)", abs(y.grad - 7) < 1e-9))
    except Exception as err:
        checks.append((f"crashed: {type(err).__name__}: {err}", False))
    for name, ok in checks:
        print("PASS" if ok else "FAIL", "-", name)
    print("=> ALL PASS" if all(ok for _, ok in checks) else "=> not yet: fix and re-run this cell")
test_value(Value)

In [ ]:
#@title ✍️ A1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("A1", what_happened, surprise)

In [ ]:
#@title Reference `Value` (open only after your test): the core class in 22 lines
class Value:
    def __init__(self, data, _children=(), _op=""):
        self.data, self.grad = float(data), 0.0
        self._prev, self._op = set(_children), _op
        self._backward = lambda: None
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")
        def _backward():                       # add copies the gradient to both parents
            self.grad += out.grad; other.grad += out.grad
        out._backward = _backward; return out
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")
        def _backward():                       # multiply swaps: each parent gets the other's value
            self.grad += other.data * out.grad; other.grad += self.data * out.grad
        out._backward = _backward; return out
    def backward(self):
        topo, seen = [], set()
        def build(v):
            if v not in seen:
                seen.add(v); [build(p) for p in v._prev]; topo.append(v)
        build(self); self.grad = 1.0
        for v in reversed(topo): v._backward()

In [ ]:
#@title Reference extras: tanh, relu, numbers on the left, minus, powers
def _tanh(self):
    t = math.tanh(self.data); out = Value(t, (self,), "tanh")
    def _backward(): self.grad += (1 - t * t) * out.grad
    out._backward = _backward; return out
def _relu(self):
    out = Value(max(0.0, self.data), (self,), "relu")
    def _backward(): self.grad += (out.data > 0) * out.grad
    out._backward = _backward; return out
def _pow(self, k):                                   # k is a plain number
    out = Value(self.data ** k, (self,), f"**{k}")
    def _backward(): self.grad += k * self.data ** (k - 1) * out.grad
    out._backward = _backward; return out
Value.tanh, Value.relu, Value.__pow__ = _tanh, _relu, _pow
Value.__radd__ = lambda self, o: self + o            # 2 + v
Value.__rmul__ = lambda self, o: self * o            # 2 * v
Value.__neg__  = lambda self: self * -1.0
Value.__sub__  = lambda self, o: self + (-o)
Value.__rsub__ = lambda self, o: (-self) + o
Value.__repr__ = lambda self: f"Value(data={self.data:.4g}, grad={self.grad:.4g})"
print("reference Value ready:", Value(2.0) * Value(-3.0) + 10, "| relu(-1) =", Value(-1.0).relu().data)
test_value(Value)

---
## Part B · Every gradient, three ways (25 min)

**Words used in this part.** *numerical gradient*: nudge one input by `h`, measure the change, divide (week 1). *PyTorch twin*: the same formula written with tensors, `requires_grad=True`, then `.backward()`. We compare in 64-bit so agreement means agreement.

The graph: `L = tanh(a*b + c) * (a + c)` with `a = 0.7, b = -1.2, c = 0.4`. Three of you will compute `dL/da`, `dL/db`, `dL/dc`.

In [ ]:
#@title 🔮 Predict B1: Your engine vs PyTorch vs nudge-and-measure (h=1e-6): will all three agree to 1e-6? Which pair differs most, and why?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("B1", prediction, confidence)

In [ ]:
import torch
vals = dict(a=0.7, b=-1.2, c=0.4)
def graph(a, b, c): return (a * b + c).tanh() * (a + c)        # works on Values AND on tensors

V = {k: Value(v) for k, v in vals.items()}
graph(**V).backward()                                                       # 1) your engine
T = {k: torch.tensor(v, dtype=torch.float64, requires_grad=True) for k, v in vals.items()}
graph(**T).backward()                                                       # 2) PyTorch twin
h = 1e-6
for k in "abc":
    bumped = dict(vals); bumped[k] += h
    num = (graph(**{kk: Value(vv) for kk, vv in bumped.items()}).data - graph(**V).data) / h   # 3) nudge
    mine, auto = V[k].grad, T[k].grad.item()
    print(f"dL/d{k}: mine {mine:+.8f} | torch {auto:+.8f} | numeric {num:+.8f} | mine-torch {abs(mine-auto):.1e}")

In [ ]:
#@title ✍️ B1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("B1", what_happened, surprise)

The engine and PyTorch agree to rounding (about 1e-16): they run the *same* algorithm. The nudge estimate is off at the sixth digit: `h` is finite. When your gradients disagree with PyTorch by a *constant factor* rather than at the last digit, suspect Part C.

---
## Part C · The accumulation bug, on purpose (15 min)

**Words used in this part.** *gradient accumulation*: `+=` in every closure, so a value used twice collects both shares. *zero_grad*: set every `.grad` back to 0 before the next backward. *stale gradient*: last step's number still in the slot.

In [ ]:
#@title 🔮 Predict C1: Two training steps WITHOUT an update and without zeroing: forward, backward, forward again, backward again (a=2, b=-3, c=10). What is a.grad after the second backward?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("C1", prediction, confidence)

In [ ]:
a, b, c = Value(2.0), Value(-3.0), Value(10.0)
def forward(): return (a * b + c) ** 2                  # a fresh graph each time, like a real step
forward().backward(); print("after 1st backward: a.grad =", a.grad)
forward().backward(); print("after 2nd backward: a.grad =", a.grad, " <- stacked, not replaced")
for p in (a, b, c): p.grad = 0.0                        # zero_grad by hand
forward().backward(); print("after zeroing + backward: a.grad =", a.grad)

L = forward(); L.backward(); L.backward()               # backward TWICE on the same graph object
print("same graph twice, leaves zeroed first? no ->", a.grad, "(the middle nodes kept their old grad too)")
w = torch.tensor(2.0, requires_grad=True)
(w * 3).backward(); (w * 3).backward()
print("PyTorch, two fresh backwards, no zeroing: w.grad =", w.grad.item(), " (true value 3)")

In [ ]:
#@title ✍️ C1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("C1", what_happened, surprise)

Zeroing the *leaves* is enough only because a real step runs a fresh forward pass, so every middle node is new. Call `backward()` twice on the same graph object and the middle nodes stack too; PyTorch refuses with *Trying to backward through the graph a second time*. That error is this cell.

In [ ]:
#@title 🔮 Predict C2: A Value whose __mul__ closure writes '=' instead of '+='. For L = e*e with e = a*b + c (a=2,b=-3,c=10): what is a.grad? (true: -24)
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("C2", prediction, confidence)

In [ ]:
class BadValue(Value):                      # identical, except the multiply closure overwrites
    def __mul__(self, other):
        other = other if isinstance(other, BadValue) else BadValue(other)
        out = BadValue(self.data * other.data, (self, other), "*")
        def _backward():
            self.grad  = other.data * out.grad     # BUG: = instead of +=
            other.grad = self.data  * out.grad
        out._backward = _backward; return out
    def __add__(self, other):
        other = other if isinstance(other, BadValue) else BadValue(other)
        out = BadValue(self.data + other.data, (self, other), "+")
        def _backward(): self.grad += out.grad; other.grad += out.grad
        out._backward = _backward; return out

a, b, c = BadValue(2.0), BadValue(-3.0), BadValue(10.0)
e = a * b + c; L = e * e; L.backward()
print(f"buggy: e.grad = {e.grad} (true 8)   a.grad = {a.grad} (true -24)   b.grad = {b.grad} (true 16)")
print("every gradient is off by the same factor: the direction is still right, so training 'sort of works'.")

In [ ]:
#@title ✍️ C2: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("C2", what_happened, surprise)

---
## Part D · Neuron → Layer → MLP, and XOR (40 min)

**Words used in this part.** *neuron*: `tanh(w·x + b)`. *layer*: several neurons on the same inputs. *MLP*: layers stacked, bend between. *hidden layer*: the middle one. *XOR*: (0,0)→0, (0,1)→1, (1,0)→1, (1,1)→0. The `nonlin` flag lets us switch the bend off to see what it buys.

In [ ]:
class Neuron:
    def __init__(self, n_in, nonlin=True):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(n_in)]
        self.b = Value(0.0); self.nonlin = nonlin
    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.tanh() if self.nonlin else act
    def parameters(self): return self.w + [self.b]

class Layer:
    def __init__(self, n_in, n_out, nonlin=True):
        self.neurons = [Neuron(n_in, nonlin) for _ in range(n_out)]
    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs
    def parameters(self): return [p for n in self.neurons for p in n.parameters()]

In [ ]:
class MLP:
    def __init__(self, n_in, sizes, nonlin=True):        # MLP(2, [4, 1]) is a 2-4-1 net
        dims = [n_in] + sizes
        self.layers = [Layer(dims[i], dims[i + 1], nonlin) for i in range(len(sizes))]
    def __call__(self, x):
        for layer in self.layers: x = layer(x)
        return x
    def parameters(self): return [p for l in self.layers for p in l.parameters()]

def train(model, xs, ys, steps, lr, every=None):
    """The five lines, plus a loss log. Returns the loss per step."""
    losses = []
    for step in range(steps):
        preds = [model(x) for x in xs]                              # 1 forward
        loss = sum((p - y) ** 2 for p, y in zip(preds, ys)) * (1 / len(ys))   # 2 loss (MSE)
        for p in model.parameters(): p.grad = 0.0                   # 3 zero
        loss.backward()                                             # 4 backward
        for p in model.parameters(): p.data -= lr * p.grad          # 5 update
        losses.append(loss.data)
        if every and step % every == 0: print(f"step {step:4d}  loss {loss.data:.4f}")
    return losses

XOR_X = [[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]]
XOR_Y = [0.0, 1.0, 1.0, 0.0]
def score(model, xs, ys): return sum((model(x).data > 0.5) == (y == 1.0) for x, y in zip(xs, ys))

In [ ]:
#@title 🔮 Predict D1: ONE tanh neuron trained on XOR (this session's step count) with MSE: what four outputs will it settle on, and what loss? (Hint: what is the best a straight line can do on average?)
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("D1", prediction, confidence)

In [ ]:
random.seed(1)
one = MLP(2, [1])                                # a single neuron
losses_one = train(one, XOR_X, XOR_Y, STEPS_XOR, lr=0.2, every=max(1, STEPS_XOR // 4))
print(f"one neuron: final loss {losses_one[-1]:.4f}; 'correct' = {score(one, XOR_X, XOR_Y)} of 4, but look at the outputs:")
for x, y in zip(XOR_X, XOR_Y): print(f"  {x} -> {one(x).data:+.3f}  (target {y:.0f})")
print("A line that must be wrong somewhere hedges: outputs near 0.5 everywhere, loss near 0.25. Ties are not 'correct'.")

In [ ]:
#@title ✍️ D1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("D1", what_happened, surprise)

In [ ]:
#@title 🔮 Predict D2: A 2-4-1 MLP with tanh vs the SAME net with the bend removed, same steps: correct counts and final losses for each?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("D2", prediction, confidence)

In [ ]:
random.seed(3)
mlp = MLP(2, [4, 1], nonlin=True)
random.seed(3)
flat = MLP(2, [4, 1], nonlin=False)              # same random start, no bend anywhere
L_mlp = train(mlp, XOR_X, XOR_Y, STEPS_XOR, lr=0.2, every=max(1, STEPS_XOR // 8))
L_flat = train(flat, XOR_X, XOR_Y, STEPS_XOR, lr=0.05)
print(f"tanh MLP : {score(mlp, XOR_X, XOR_Y)} of 4, final loss {L_mlp[-1]:.4f}")
print(f"no bend  : {score(flat, XOR_X, XOR_Y)} of 4, final loss {L_flat[-1]:.4f}")
plt.plot(L_mlp, label="2-4-1 tanh"); plt.plot(L_flat, label="2-4-1 no bend"); plt.plot(losses_one, label="1 neuron")
plt.yscale("log"); plt.xlabel("step"); plt.ylabel("MSE"); plt.legend(); plt.title("XOR loss curves"); plt.show()

In [ ]:
#@title ✍️ D2: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("D2", what_happened, surprise)

Two linear layers are one matrix: `W2 @ W1`. The "no bend" curve sits at the same 0.25 floor as one neuron (hedging at 0.5), no matter how wide the hidden layer. Watch the tanh curve for a **plateau** near 0.25 before the drop: the hidden units sit at that same hedge until they take different jobs. If yours has not dropped yet, that is the plateau: with `SMOKE = False` it gets the steps it needs.

---
## Part E · Moons: a loss curve, a decision boundary, a PyTorch twin (45 min)

**Words used in this part.** *moons*: two interleaved half-circles, the standard 2-D toy set (no straight line separates them). *decision boundary*: where the model's output crosses 0.5. *twin*: the same 17-or-so numbers loaded into tensors so PyTorch can run the identical forward pass.

In [ ]:
def make_moons(n, noise=0.15, seed=0):
    g = np.random.default_rng(seed); t = g.uniform(0, np.pi, n // 2)
    top = np.stack([np.cos(t), np.sin(t)], 1)                   # upper half-circle, class 0
    bot = np.stack([1 - np.cos(t), 0.5 - np.sin(t)], 1)          # lower one, shifted, class 1
    X = np.concatenate([top, bot]) + noise * g.standard_normal((n // 2 * 2, 2))
    y = np.concatenate([np.zeros(n // 2), np.ones(n // 2)])
    return X, y

MX, My = make_moons(N_MOON)
moon_xs = [[float(a), float(b)] for a, b in MX]
moon_ys = [float(v) for v in My]
plt.scatter(MX[:, 0], MX[:, 1], c=My, cmap="coolwarm", s=18); plt.title(f"moons, {len(My)} points"); plt.show()

In [ ]:
#@title 🔮 Predict E1: A 2-8-1 tanh MLP on moons for this session's step count: roughly what final MSE, and what shape will the boundary be (a line, one curve, two blobs)?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("E1", prediction, confidence)

In [ ]:
random.seed(7)
net = MLP(2, [8, 1])
t0 = time.time()
L_moon = train(net, moon_xs, moon_ys, STEPS_MOON, LR, every=max(1, STEPS_MOON // 5))
print(f"{STEPS_MOON} steps in {time.time() - t0:.1f} s | final loss {L_moon[-1]:.4f} | "
      f"{score(net, moon_xs, moon_ys)} of {len(moon_ys)} correct")
plt.plot(L_moon); plt.yscale("log"); plt.xlabel("step"); plt.ylabel("MSE"); plt.title("moons loss curve"); plt.show()

In [ ]:
gx, gy = np.meshgrid(np.linspace(-1.6, 2.6, 36), np.linspace(-1.2, 1.7, 30))
Z = np.array([net([float(a), float(b)]).data for a, b in zip(gx.ravel(), gy.ravel())]).reshape(gx.shape)
plt.contourf(gx, gy, Z, levels=[-9, 0.5, 9], colors=["#9db4ff", "#ffb08a"], alpha=0.6)
plt.contour(gx, gy, Z, levels=[0.5], colors="k")
plt.scatter(MX[:, 0], MX[:, 1], c=My, cmap="coolwarm", s=18, edgecolors="k", linewidths=0.3)
plt.title("decision boundary (output = 0.5)"); plt.show()

In [ ]:
#@title ✍️ E1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("E1", what_happened, surprise)

In [ ]:
#@title 🔮 Predict E2: Load the SAME weights into a PyTorch 2-8-1 net and run one forward+backward on the moons set. Max |grad difference| vs your engine: 0, ~1e-9, or ~1e-3?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("E2", prediction, confidence)

In [ ]:
def torch_twin(model):
    """Copy each layer's weights into tensors and run the identical forward pass in float64."""
    Ws = [torch.tensor([[w.data for w in n.w] for n in l.neurons], dtype=torch.float64, requires_grad=True)
          for l in model.layers]
    bs = [torch.tensor([n.b.data for n in l.neurons], dtype=torch.float64, requires_grad=True) for l in model.layers]
    return Ws, bs
def torch_forward(Ws, bs, X):
    h = X
    for W, b in zip(Ws, bs): h = torch.tanh(h @ W.T + b)
    return h.squeeze(1)

Ws, bs = torch_twin(net)
Xt, yt = torch.tensor(MX, dtype=torch.float64), torch.tensor(My, dtype=torch.float64)
loss_t = ((torch_forward(Ws, bs, Xt) - yt) ** 2).mean(); loss_t.backward()

preds = [net(x) for x in moon_xs]                                  # one more step of yours, no update
loss_v = sum((p - y) ** 2 for p, y in zip(preds, moon_ys)) * (1 / len(moon_ys))
for p in net.parameters(): p.grad = 0.0
loss_v.backward()
mine = torch.tensor([[[w.grad for w in n.w] for n in l.neurons] for l in net.layers][0], dtype=torch.float64)
print(f"loss: mine {loss_v.data:.10f} | torch {loss_t.item():.10f}")
print("layer-1 weight grads: max |mine - torch| =", (mine - Ws[0].grad).abs().max().item())
print("allclose:", torch.allclose(mine, Ws[0].grad, atol=1e-9))

In [ ]:
#@title ✍️ E2: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("E2", what_happened, surprise)

Same numbers, same gradients. PyTorch is your engine with tensors instead of scalars, C++ instead of Python, and a GPU when you ask. From week 3 you use it directly.

---
## Part F · Stretch: ReLU vs tanh, and a learning-rate sweep (15 min)

**Words used in this part.** *ReLU*: `max(0, x)`, slope 0 or 1. *learning-rate sweep*: the same run at several step sizes, loss curves overlaid.

In [ ]:
#@title 🔮 Predict F1: Same 2-8-1 net on moons with ReLU hidden units instead of tanh, same steps and lr: faster, slower, or stuck? And which of lr 0.01 / 0.05 / 0.2 will show a plateau?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("F1", prediction, confidence)

In [ ]:
class ReluMLP(MLP):
    def __init__(self, n_in, sizes):
        super().__init__(n_in, sizes)
        for l in self.layers[:-1]:
            for n in l.neurons: n.nonlin = False      # hidden layers: linear, then relu below
    def __call__(self, x):
        for l in self.layers[:-1]: x = [v.relu() for v in l(x)]
        return self.layers[-1](x)                     # output neuron keeps its tanh

random.seed(7); r = ReluMLP(2, [8, 1])
L_relu = train(r, moon_xs, moon_ys, STEPS_MOON, LR)
print(f"relu: final loss {L_relu[-1]:.4f}, {score(r, moon_xs, moon_ys)} of {len(moon_ys)} | tanh was {L_moon[-1]:.4f}")
for lr in [0.01, 0.05, 0.2]:
    random.seed(7); m = MLP(2, [8, 1])
    plt.plot(train(m, moon_xs, moon_ys, STEPS_MOON, lr), label=f"tanh lr={lr}")
plt.plot(L_relu, "--", label=f"relu lr={LR}"); plt.yscale("log"); plt.legend(); plt.xlabel("step"); plt.ylabel("MSE"); plt.show()

In [ ]:
#@title ✍️ F1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("F1", what_happened, surprise)

---
## Wrap up

Run the cell below, copy the table into your insight ledger next to `core-calc` (and the Part D line next to `move-discrete-continuous`), and mark the biggest surprise.
Then on the laptop:

```
python track.py done week-02/build --rating <1-5> --minutes 180 --note "biggest surprise"
python gen_week.py 3
```

In [ ]:
reveal()